# Data Preprocessing

This notebook demonstrates two preprocessing workflows:

1. **Blur analysis** — generate per-image blur heatmaps to assess focus quality
2. **Z-stack → 3D conversion** — combine 2D optical sections into 3D volumetric TIFFs

Blur heatmaps assign a per-pixel sharpness score to each image (or z-slice). High values = sharp; low values = blurry. These are computed with a patchwise Laplacian variance.

Every section includes a **self-contained demo** using synthetic data that runs without any external files.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

# Ensure the project root (parent of notebooks/) is on sys.path so `src` is importable.
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

In [ ]:
import tempfile

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
import tifffile as tiff

from src.preprocessing.blur_analysis import generate_blur_heatmap_batch
from src.utils.blur_measure import generate_blur_heatmap
from src.utils.file_utils import BlurFileHandler
from src.utils.conversion import combine_2d_to_3d

## 1. Run on your own data

Set the paths below and run the cell.

### 1.1 Single image mode

In [29]:
image_path = "../data/Plate 2426_preprocessed_3D/p2426_B02_BF_3d.tif"
# output_path = '../data/Plate 2426_preprocessed_3D/blur_heatmaps/t1_B02_s1_w1_blur_heatmap.tif'  # Optional: specify output path

In [30]:
# Generate blur heatmap
blur_heatmap = generate_blur_heatmap(
    image_path,
    # blur_path=output_path,
    # patch_size=32,
    # stride_size=16,
    # normalize=True,
    # center_values=True,
)

### Load blur heatmap from disk

In [ ]:
blur_path = "/Users/serenasritharan/Documents/single_cell_docs/pMF5V1_C09_t11_BF_3d_blur_heatmap.tif"  # Path to your 3D image
blur_heatmap = tiff.imread(blur_path)

### Visualization

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

heatmap_img = blur_heatmap  # shape: (Z, Y, X)
title = "Blur Heatmap"

if heatmap_img.ndim == 2:
    # Single 2D image – display directly
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(heatmap_img, cmap="hot", interpolation="nearest")
    ax.set_title(title)
    ax.axis("off")
    plt.colorbar(im, ax=ax)
    plt.show()
elif heatmap_img.ndim == 3:
    z_slices = heatmap_img.shape[0]

    slider = widgets.IntSlider(
        value=0, min=0, max=z_slices - 1, step=1,
        description=f"Z slice (0–{z_slices - 1}):",
        style={"description_width": "initial"},
        continuous_update=True,
        layout=widgets.Layout(width="500px"),
    )
    cmap_dropdown = widgets.Dropdown(
        value="hot",
        options=["hot", "viridis", "magma", "inferno", "gray", "plasma"],
        description="Colormap:",
    )

    out = widgets.Output()

    def show_slice(z: int, cmap: str) -> None:
        with out:
            out.clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(8, 6))
            im = ax.imshow(heatmap_img[z], cmap=cmap, interpolation="nearest")
            ax.set_title(f"{title} — Z slice {z + 1}/{z_slices}")
            ax.axis("off")
            plt.colorbar(im, ax=ax)
            plt.tight_layout()
            plt.show()

    widgets.interactive(show_slice, z=slider, cmap=cmap_dropdown)
    display(widgets.VBox([widgets.HBox([slider, cmap_dropdown]), out]))
    show_slice(0, "hot")  # render initial frame
else:
    raise ValueError(f"Unsupported heatmap shape: {heatmap_img.shape}")


## Batch mode

In [ ]:
input_dir = '../data/Plate 2426_preprocessed_3D_split/'
output_dir = '../data/Plate 2426_preprocessed_2D_split/blur_heatmaps/'
pattern = '**/*_BF_3d.tif'  # Adjust pattern as needed



In [ ]:
results = generate_blur_heatmap_batch(
    input_dir,
    output_dir=output_dir,
    pattern=pattern,
    patch_size=32,
    stride_size=16,
    normalize=True,
    center_values=True,
)

Generating blur heatmaps:   0%|          | 0/80 [00:00<?, ?it/s]

Generating blur heatmaps: 100%|██████████| 80/80 [01:44<00:00,  1.30s/it]


---
## 2. Synthetic data samples


### 2.1 Single-image demo

In [ ]:
# ── Create a synthetic 3-D z-stack (Z=5, Y=128, X=128) ──────────────────────
rng = np.random.default_rng(42)

z_stack = np.zeros((5, 128, 128), dtype=np.uint16)
for z in range(5):
    # Sharp slices in the middle, blurrier near the edges
    noise_scale = 3000 + abs(z - 2) * 2000
    z_stack[z] = rng.integers(0, noise_scale, (128, 128), dtype=np.uint16)

with tempfile.TemporaryDirectory() as tmp:
    img_path = Path(tmp) / "demo_stack.tif"
    tiff.imwrite(str(img_path), z_stack)

    heatmap = generate_blur_heatmap(
        img_path,
        patch_size=32,
        stride_size=8,
        normalize=True,
    )

print(f"Input stack shape : {z_stack.shape}")
print(f"Heatmap shape     : {heatmap.shape}")
print(f"Value range       : [{heatmap.min():.3f}, {heatmap.max():.3f}]")

### 2.2 Visualize heatmap

For a 2-D image a static plot is shown; for a 3-D stack an interactive z-slider is displayed.

In [ ]:
%matplotlib inline

# Replace `heatmap` with any loaded heatmap array.
heatmap_img = heatmap
title = "Blur Heatmap"

if heatmap_img.ndim == 2:
    fig, ax = plt.subplots(figsize=(7, 5))
    im = ax.imshow(heatmap_img, cmap="hot", interpolation="nearest")
    ax.set_title(title)
    ax.axis("off")
    plt.colorbar(im, ax=ax)
    plt.show()

elif heatmap_img.ndim == 3:
    z_slices = heatmap_img.shape[0]

    slider = widgets.IntSlider(
        value=0, min=0, max=z_slices - 1, step=1,
        description=f"Z (0–{z_slices - 1}):",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="480px"),
    )
    cmap_dd = widgets.Dropdown(
        value="hot",
        options=["hot", "viridis", "magma", "inferno", "gray", "plasma"],
        description="Colormap:",
    )
    out = widgets.Output()

    def _show_slice(z: int, cmap: str) -> None:
        with out:
            out.clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(7, 5))
            im = ax.imshow(heatmap_img[z], cmap=cmap, interpolation="nearest")
            ax.set_title(f"{title} — slice {z + 1}/{z_slices}")
            ax.axis("off")
            plt.colorbar(im, ax=ax)
            plt.tight_layout()
            plt.show()

    widgets.interactive(_show_slice, z=slider, cmap=cmap_dd)
    display(widgets.VBox([widgets.HBox([slider, cmap_dd]), out]))
    _show_slice(0, "hot")

else:
    raise ValueError(f"Unsupported heatmap shape: {heatmap_img.shape}")

### 2.3 Load a heatmap saved to disk

In [ ]:
# Demonstrate round-trip: save → reload.
with tempfile.TemporaryDirectory() as tmp:
    img_path = Path(tmp) / "demo_stack.tif"
    blur_path = Path(tmp) / "demo_stack_blur.tif"
    tiff.imwrite(str(img_path), z_stack)

    _ = generate_blur_heatmap(img_path, blur_path=blur_path, normalize=True)
    reloaded = tiff.imread(str(blur_path))

print(f"Reloaded heatmap shape : {reloaded.shape}, dtype: {reloaded.dtype}")

### 2.4 Batch processing (demo)

In [ ]:
# Create a small in-memory dataset and run batch heatmap generation.
with tempfile.TemporaryDirectory() as tmp:
    input_dir = Path(tmp) / "images"
    output_dir = Path(tmp) / "heatmaps"
    input_dir.mkdir()

    # Write three synthetic 3-D stacks.
    for i in range(3):
        img = rng.integers(0, 5000, (4, 64, 64), dtype=np.uint16)
        tiff.imwrite(str(input_dir / f"p2126_A0{i+1}_t1_BF_3d.tif"), img)

    results = generate_blur_heatmap_batch(
        input_dir=str(input_dir),
        output_dir=str(output_dir),
        pattern="**/*.tif",
        patch_size=16,
        stride_size=8,
        normalize=True,
        overwrite=False,
    )

    print(f"Processed {len(results)} images.")
    for src, dst in results.items():
        print(f"  {Path(src).name}  →  {Path(dst).name}")

#### Overwrite behaviour

`overwrite=False` (default) skips images whose heatmap file already exists — useful for resuming interrupted runs. Set `overwrite=True` to recompute everything.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    input_dir = Path(tmp) / "images"
    output_dir = Path(tmp) / "heatmaps"
    input_dir.mkdir()
    tiff.imwrite(str(input_dir / "img.tif"), rng.integers(0, 4000, (4, 64, 64), dtype=np.uint16))

    # First pass — computes and saves the heatmap.
    generate_blur_heatmap_batch(input_dir=str(input_dir), output_dir=str(output_dir), normalize=True)
    heatmap_files_after_first = list(output_dir.glob("*.tif"))
    print(f"Files after first pass : {len(heatmap_files_after_first)}")

    # Second pass with overwrite=False — skips existing.
    results_skip = generate_blur_heatmap_batch(
        input_dir=str(input_dir), output_dir=str(output_dir), overwrite=False, normalize=True
    )
    print(f"Results returned (skip) : {len(results_skip)}   ← same file, no recompute")

    # Third pass with overwrite=True — recomputes.
    results_overwrite = generate_blur_heatmap_batch(
        input_dir=str(input_dir), output_dir=str(output_dir), overwrite=True, normalize=True
    )
    print(f"Results returned (overwrite) : {len(results_overwrite)}   ← recomputed")

---
## 3. Z-Stack → 3D Conversion

2D microscopy acquisitions produce one TIFF per optical section (z-slice). This step stacks related slices into a single 3D volume, which is required as input for blur analysis and cell segmentation.

Files are grouped by matching a regular-expression **pattern** against each filename. The default pattern captures:
- **base name** — everything before the first `_z`
- **z-index** — the integer after `_z`
- **channel suffix** — optional `_BF` or `_Cells` (or similar) before the extension

### 3.1 Synthetic demo (two channels, two wells)

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    input_dir  = Path(tmp) / "slices"
    output_dir = Path(tmp) / "volumes"
    input_dir.mkdir()

    rng_local = np.random.default_rng(0)
    wells    = ["A01", "B02"]
    channels = ["BF", "Cells"]

    # Write 2D z-slices: p2126_A01_t1_z1_BF.tif, ..., p2126_B02_t1_z3_Cells.tif
    for well in wells:
        for ch in channels:
            for z in range(1, 4):  # z1, z2, z3
                img = rng_local.integers(0, 4000, (64, 64), dtype=np.uint16)
                tiff.imwrite(str(input_dir / f"p2126_{well}_t1_z{z}_{ch}.tif"), img)

    combine_2d_to_3d(
        input_dir=str(input_dir),
        output_dir=str(output_dir),
        pattern=r"(.+?)_z(\d+)_(BF|Cells)\.(tif)",
    )

    volumes = sorted(output_dir.glob("*.tif"))
    print(f"\nCreated {len(volumes)} 3D volumes:")
    for v in volumes:
        vol = tiff.imread(str(v))
        print(f"  {v.name}  →  shape {vol.shape}  dtype {vol.dtype}")

### 3.2 z-axis filtering with `z_min` / `z_max`

Microscopes often save a z0 slice that is a 2D projection rather than an optical section. `z_min=1` (the default) skips that slice. Use `z_max` to cap the upper end.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    input_dir = Path(tmp) / "slices"
    input_dir.mkdir()

    rng_local = np.random.default_rng(1)
    # Create z0 (projection) + z1–z3 (optical sections)
    for z in range(4):
        img = rng_local.integers(0, 4000, (64, 64), dtype=np.uint16)
        tiff.imwrite(str(input_dir / f"sample_z{z}_BF.tif"), img)

    for z_min, label in [(1, "z_min=1 (default — skips z0)"), (0, "z_min=0  (includes z0)")]:
        out = Path(tmp) / f"out_{z_min}"
        combine_2d_to_3d(
            input_dir=str(input_dir),
            output_dir=str(out),
            pattern=r"(.+?)_z(\d+)_(BF|Cells)\.(tif)",
            z_min=z_min,
        )
        vol = tiff.imread(str(out / "sample_BF_3d.tif"))
        print(f"  {label}  →  shape {vol.shape}  ({vol.shape[0]} slices)")

### 3.3 Run on your own data

Set the paths and the regex pattern that matches your filename convention, then run the cell.

In [ ]:
# USER: set these paths ────────────────────────────────────────────────────────
INPUT_DIR  = "/path/to/your/2d_slices"   # Directory containing 2D .tif files
OUTPUT_DIR = "/path/to/your/3d_volumes"  # Will be created if it does not exist
PATTERN    = r"(.+?)_z(\d+)(?:_(BF|Cells))?\.(tif|tiff)"  # default pattern
Z_MIN      = 1      # set to 0 to include the z0 projection slice
Z_MAX      = None   # set to an integer to cap the upper z-index
RECURSIVE  = True   # search subdirectories
# ─────────────────────────────────────────────────────────────────────────────

if INPUT_DIR == "/path/to/your/2d_slices":
    print("Please set INPUT_DIR and OUTPUT_DIR before running this cell.")
else:
    combine_2d_to_3d(
        input_dir=INPUT_DIR,
        output_dir=OUTPUT_DIR,
        pattern=PATTERN,
        recursive=RECURSIVE,
        z_min=Z_MIN,
        z_max=Z_MAX,
    )